<a href="https://colab.research.google.com/github/sagnikghosal007/3rdSem_Pythonprojects/blob/main/LLM_FineTuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## **1. Enviroment Setup**

In [1]:
# install the required libraries
!pip install -q \
  "transformers==5.0.0" \
  "peft==0.18.1" \
  "accelerate==1.13.0" \
  "datasets==4.8.4" \
  "trl==1.1.0" \
  "sentencepiece==0.2.1" \
  "protobuf==5.29.6"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 42.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 557.0/557.0 kB 23.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 20.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 678.0/678.0 kB 30.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 30.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 11.1 MB/s eta 0:00:00


In [2]:
import sys
import torch

print("Python   : ", sys.version.split()[0])
print("PyTorch  :", torch.__version__)


# check GPU availability



Python   :  3.12.13
PyTorch  : 2.11.0+cu128


In [3]:
import sys
import torch # pre-installed in colab env

print("Python     :", sys.version.split()[0])
print("PyTorch      :", torch.__version__)

# check GPU availability
print("CUDA avail :", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU name   :", torch.cuda.get_device_name(0))
    props = torch.cuda.get_device_properties(0)
    print(f"GPU memory : {props.total_memory / 1024**3:.2f} GB")

# Expected on Colab free tier: Tesla T4, ~14.5 GB

Python     : 3.12.13
PyTorch      : 2.11.0+cu128
CUDA avail : True
GPU name   : Tesla T4
GPU memory : 14.56 GB


## 2. Load the base model in FP16

We load the model in half precision (torch.float16). On T4 this is the sweet spot - faster than FP32

In [4]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=torch.float16,
    device_map="auto",
)

model.config.use_cache = False          # required for gradient checkpointing later
model.config.pretraining_tp = 1

n_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {n_params/1e6:.1f} M")
print(f"Memory footprint: {model.get_memory_footprint()/1024**3:.2f} GB")

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  500kB            

tokenizer.model: downloading bytes:           |  0.00B            

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.20GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Total parameters: 1100.0 M
Memory footprint: 2.05 GB


  ##  3 . Define a Prompt format and test the BASE model


  We use TinyLlama's chat template. Task -> text to SQL . Given a table schema and a question , produce SQL query.

In [5]:
def build_prompt(schema: str, question: str) -> str:
    system = (
        "You are a SQL assistant. Given a table schema and a question, "
        "reply with ONLY the SQL query, nothing else."
    )
    user = f"Schema:\n{schema}\n\nQuestion: {question}"
    messages = [
        {"role": "system", "content": system},
        {"role": "user",   "content": user},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

In [6]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

# Re-initialize tokenizer here to ensure it's defined within this cell's scope
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

def build_prompt(tokenizer_obj, schema: str, question: str) -> str:
    system = (
        "You are a SQL assistant. Given a table schema and a question, "
        "reply with ONLY the SQL query, nothing else."
    )
    user = f"Schema:\n{schema}\n\nQuestion: {question}"
    messages = [
        {"role": "system", "content": system},
        {"role": "user",   "content": user},
    ]
    return tokenizer_obj.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

# testing build_prompt function
test_prompt_1 = build_prompt(
    tokenizer,
    schema="CREATE TABLE employees (id INT, name TEXT, department TEXT, salary INT);",
    question="List the names of employees in the Engineering department earning more than 100000."
)

test_prompt_1

'<|system|>\nYou are a SQL assistant. Given a table schema and a question, reply with ONLY the SQL query, nothing else.</s>\n<|user|>\nSchema:\nCREATE TABLE employees (id INT, name TEXT, department TEXT, salary INT);\n\nQuestion: List the names of employees in the Engineering department earning more than 100000.</s>\n<|assistant|>\n'

In [7]:
print(test_prompt_1)

<|system|>
You are a SQL assistant. Given a table schema and a question, reply with ONLY the SQL query, nothing else.</s>
<|user|>
Schema:
CREATE TABLE employees (id INT, name TEXT, department TEXT, salary INT);

Question: List the names of employees in the Engineering department earning more than 100000.</s>
<|assistant|>



In [8]:
@torch.no_grad()
def generate(prompt: str, max_new_tokens: int = 120) -> str :
  inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

  output_ids = model.generate(
    **inputs,
    max_new_tokens=max_new_tokens,
    do_sample=False,
    pad_token_id=tokenizer.pad_token_id,
    eos_token_id=tokenizer.eos_token_id,
  )

  # Slice only newly generated tokens
  input_length = inputs["input_ids"].shape[1]
  new_tokens = output_ids[0][input_length:]

  return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


In [9]:
output = generate(test_prompt_1)
print(output)

To answer this question, you can use the following SQL query:

```
SELECT name
FROM employees
WHERE department = 'Engineering'
AND salary > 100000;
```

This query will return a list of names of employees in the Engineering department who have a salary greater than 100,000.


In [10]:
# Three probe prompts we will reuse AFTER fine-tuning for direct comparison.
PROBES = [
    {
        "schema":  "CREATE TABLE employees (id INT, name TEXT, department TEXT, salary INT);",
        "question":"List the names of employees in the Engineering department earning more than 100000.",
    },
    {
        "schema":  "CREATE TABLE orders (order_id INT, customer_id INT, amount FLOAT, order_date DATE);",
        "question":"What is the total order amount per customer in 2024?",
    },
    {
        "schema":  "CREATE TABLE movies (title TEXT, year INT, rating FLOAT, genre TEXT);",
        "question":"Show the top 5 highest rated horror movies released after 2015.",
    },
]

print("="*70)
print("BASE MODEL (before fine-tuning)")
print("="*70)
base_outputs = []
for i, p in enumerate(PROBES, 1):
    prompt = build_prompt(tokenizer, p["schema"], p["question"])
    ans = generate(prompt)
    base_outputs.append(ans)
    print(f"\n--- Probe {i} ---")
    print("Q:", p["question"])
    print("A:", ans)

BASE MODEL (before fine-tuning)

--- Probe 1 ---
Q: List the names of employees in the Engineering department earning more than 100000.
A: To answer this question, you can use the following SQL query:

```
SELECT name
FROM employees
WHERE department = 'Engineering'
AND salary > 100000;
```

This query will return a list of names of employees in the Engineering department who have a salary greater than 100,000.

--- Probe 2 ---
Q: What is the total order amount per customer in 2024?
A: To answer the question, you need to use the `SUM` function to calculate the total order amount per customer in 2024. Here's the SQL query:

```sql
SELECT SUM(amount) AS total_order_amount_per_customer
FROM orders
WHERE year(order_date) = 2024
GROUP BY customer_id;
```

This query will return a single row with the total order amount per customer in 2024.

--- Probe 3 ---
Q: Show the top 5 highest rated horror movies released after 2015.
A: To show the top 5 highest rated horror movies released after 2015, 

## 4. Load and Prepare the Dataset
We use b-mc2/sql-create-context - a compact text-to-SQL dataset with
(question, context, answer) triples. We'll take a small slice so training finishes in a few minutes on T4.

In [11]:
from datasets import load_dataset
raw = load_dataset("b-mc2/sql-create-context", split="train")
print("Full dataset size:", len(raw))
print("Example row    :", raw[0])

# Keep it small for a fast, visible demo on T4
raw = raw.shuffle(seed=42).select(range(3000))
split = raw.train_test_split(test_size=0.05, seed=42)
train_ds, eval_ds = split["train"], split["test"]
print("Train:", len(train_ds), " Eval:", len(eval_ds))

README.md:   0%|          | 0.00/4.43k [00:00<?, ?B/s]

sql_create_context_v4.json: reconstructing file:   0%|          |  0.00B / 21.8MB            

sql_create_context_v4.json: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/78577 [00:00<?, ? examples/s]

Full dataset size: 78577
Example row    : {'answer': 'SELECT COUNT(*) FROM head WHERE age > 56', 'question': 'How many heads of the departments are older than 56 ?', 'context': 'CREATE TABLE head (age INTEGER)'}
Train: 2850  Eval: 150


In [12]:
train_ds[0]

{'answer': 'SELECT first_name, last_name, hire_date FROM employees WHERE department_id = (SELECT department_id FROM employees WHERE first_name = "Clara")',
 'question': 'display the employee name ( first name and last name ) and hire date for all employees in the same department as Clara.',
 'context': 'CREATE TABLE employees (first_name VARCHAR, last_name VARCHAR, hire_date VARCHAR, department_id VARCHAR)'}

In [13]:
train_ds[:3]

{'answer': ['SELECT first_name, last_name, hire_date FROM employees WHERE department_id = (SELECT department_id FROM employees WHERE first_name = "Clara")',
  'SELECT longitude FROM table_16799784_8 WHERE name = "Razia Patera"',
  'SELECT opponent FROM table_21091145_1 WHERE black_knights_points = 27'],
 'question': ['display the employee name ( first name and last name ) and hire date for all employees in the same department as Clara.',
  'What is the longitude of the feature named Razia Patera? ',
  'Name the opponent for black knights points 27'],
 'context': ['CREATE TABLE employees (first_name VARCHAR, last_name VARCHAR, hire_date VARCHAR, department_id VARCHAR)',
  'CREATE TABLE table_16799784_8 (longitude VARCHAR, name VARCHAR)',
  'CREATE TABLE table_21091145_1 (opponent VARCHAR, black_knights_points VARCHAR)']}

In [14]:
print("Schema:", train_ds[0]["context"])
print("="*70)
print("Question:", train_ds[0]["question"])
print("="*70)
print("Answer:", train_ds[0]["answer"])
print("="*70)

Schema: CREATE TABLE employees (first_name VARCHAR, last_name VARCHAR, hire_date VARCHAR, department_id VARCHAR)
Question: display the employee name ( first name and last name ) and hire date for all employees in the same department as Clara.
Answer: SELECT first_name, last_name, hire_date FROM employees WHERE department_id = (SELECT department_id FROM employees WHERE first_name = "Clara")


In [15]:
def format_example(row):
    system = (
        "You are a SQL assistant. Given a table schema and a question, "
        "reply with ONLY the SQL query, nothing else."
    )
    user      = f"Schema:\n{row['context']}\n\nQuestion: {row['question']}"
    assistant = row["answer"]
    messages = [
        {"role": "system",    "content": system},
        {"role": "user",      "content": user},
        {"role": "assistant", "content": assistant},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False)
    return {"text": text}

train_ds = train_ds.map(format_example, remove_columns=train_ds.column_names)
eval_ds  = eval_ds.map (format_example, remove_columns=eval_ds.column_names)

print("\n--- Formatted training example ---\n")
print(train_ds[0]["text"][:800])# underatand the dataset
train_ds[0].keys()

Map:   0%|          | 0/2850 [00:00<?, ? examples/s]

Map:   0%|          | 0/150 [00:00<?, ? examples/s]


--- Formatted training example ---

<|system|>
You are a SQL assistant. Given a table schema and a question, reply with ONLY the SQL query, nothing else.</s>
<|user|>
Schema:
CREATE TABLE employees (first_name VARCHAR, last_name VARCHAR, hire_date VARCHAR, department_id VARCHAR)

Question: display the employee name ( first name and last name ) and hire date for all employees in the same department as Clara.</s>
<|assistant|>
SELECT first_name, last_name, hire_date FROM employees WHERE department_id = (SELECT department_id FROM employees WHERE first_name = "Clara")</s>



dict_keys(['text'])

## 5. Attach LoRA Adapters
LoRA inserts low-rank trainable matrices into specific linear layers(attention projections) while freezing the base weights . For Llama-family models the common target modules are q_proj, k_proj, v_proj, o_proj (and optionally the MLP projections).

In [16]:
from peft import LoraConfig , get_peft_model

# Enable gradient checkpointing to save VRAM during backward pass
model.gradient_checkpointing_enable()
model.enable_input_require_grads() # needed beacause base params are frozen


lora_config = LoraConfig(
    r=32,         # rank
    lora_alpha=32,   # scaling factor (alpha/r = 2.0)
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj","k_proj","v_proj","o_proj"],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
# You should see something like: trainable: ~4M / all: ~1.1B  (< 0.5%)


trainable params: 9,011,200 || all params: 1,109,059,584 || trainable%: 0.8125


## 6. Train

We use SFTTrainer from TRL.
Settings chosen for T4(16 GB):
     

*   per_device_train_batch_size =2 , gradient_accumulation_steps=8 → effective batch 16
*   max_seq_length=512


*   fp16=True (T4 supports FP16, not BF16)
*   1 epoch over 3000 examples ≈ ~5–8 minutes





In [17]:
# Peak VRAM used during training — should stay under T4's 15.8 GB
import torch
print(f"Peak GPU memory allocated: {torch.cuda.max_memory_allocated()/1024**3:.2f} GB")

Peak GPU memory allocated: 2.09 GB


In [18]:
from trl import SFTTrainer, SFTConfig

OUTPUT_DIR = "./tinyllama-sql-lora"

sft_config = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=1,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,
    gradient_checkpointing=True,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=10,
    optim="adamw_torch",
    fp16=True,
    bf16=False,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=50,
    save_strategy="epoch",
    report_to="none",
)

# Pre-tokenize so we don't depend on SFTTrainer's text-field handling
def tokenize(batch):
    out = tokenizer(
        batch["text"],
        truncation=True,
        max_length=512,
        padding=False,
    )
    return out

train_tok = train_ds.map(tokenize, batched=True, remove_columns=["text"])
eval_tok  = eval_ds.map (tokenize, batched=True, remove_columns=["text"])

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_tok,
    eval_dataset=eval_tok,
    processing_class=tokenizer,
)

trainer.train()

Map:   0%|          | 0/2850 [00:00<?, ? examples/s]

Map:   0%|          | 0/150 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


Step,Training Loss,Validation Loss
50,0.639135,0.626124
100,0.598224,0.580603
150,0.567131,0.569958


TrainOutput(global_step=179, training_loss=0.705382251206723, metrics={'train_runtime': 430.4415, 'train_samples_per_second': 6.621, 'train_steps_per_second': 0.416, 'total_flos': 2475041758814208.0, 'train_loss': 0.705382251206723})

In [19]:
# Peak VRAM used during training — should stay under T4's 15.8 GB
import torch
print(f"Peak GPU memory allocated: {torch.cuda.max_memory_allocated()/1024**3:.2f} GB")

Peak GPU memory allocated: 2.45 GB


## SETConfig Parameters Explained

- `output_dir=OUTPUT_DIR`  
  Folder where the fine-tuned model, checkpoints, and logs will be saved.

- `num_train_epochs=1`  
  Number of times the model sees the full training dataset.

- `per_device_train_batch_size=2`  
  Number of training samples processed at once on each device.

- `per_device_eval_batch_size=2`  
  Number of evaluation samples processed at once on each device.

- `gradient_accumulation_steps=8`  
  Accumulates gradients for 8 steps before updating weights, which gives an effective larger batch size.

- `gradient_checkpointing=True`  
  Saves GPU memory by recomputing some activations during backpropagation.

- `learning_rate=2e-4`  
  Controls how big each weight update is during training.

- `lr_scheduler_type="cosine"`  
  Gradually changes the learning rate using a cosine decay schedule.

- `warmup_steps=10`  
  Slowly increases the learning rate for the first 10 steps to make training stable.

- `optim="adamw_torch"`  
  Optimizer used to update model weights, here it is AdamW from PyTorch.

- `fp16=True`  
  Uses 16-bit floating point precision to reduce memory usage and speed up training.

- `bf16=False`  
  Disables bfloat16 precision.

- `logging_steps=10`  
  Logs training details every 10 steps.

- `eval_strategy="steps"`  
  Runs evaluation during training based on step intervals.

- `eval_steps=50`  
  Evaluates the model every 50 steps.

- `save_strategy="epoch"`  
  Saves a checkpoint after each epoch.

- `report_to="none"`  
  Disables reporting to tools like WandB or TensorBoard.

## 7. Compare: AFTER Fine-Tuning

In [20]:
model.config.use_cache = True
model.eval()

print("="*70)
print("FINE-TUNED MODEL (after LoRA)")
print("="*70)
for i, p in enumerate(PROBES, 1):
    prompt = build_prompt(tokenizer, p["schema"], p["question"])
    ans = generate(prompt)
    print(f"\n--- Probe {i} ---")
    print("Q:     ", p["question"])
    print("BEFORE:", base_outputs[i-1])
    print("="*70)
    print("AFTER :", ans)
    print("="*70)

FINE-TUNED MODEL (after LoRA)

--- Probe 1 ---
Q:      List the names of employees in the Engineering department earning more than 100000.
BEFORE: To answer this question, you can use the following SQL query:

```
SELECT name
FROM employees
WHERE department = 'Engineering'
AND salary > 100000;
```

This query will return a list of names of employees in the Engineering department who have a salary greater than 100,000.
AFTER : SELECT name FROM employees WHERE department = "engineering" AND salary > 100000

--- Probe 2 ---
Q:      What is the total order amount per customer in 2024?
BEFORE: To answer the question, you need to use the `SUM` function to calculate the total order amount per customer in 2024. Here's the SQL query:

```sql
SELECT SUM(amount) AS total_order_amount_per_customer
FROM orders
WHERE year(order_date) = 2024
GROUP BY customer_id;
```

This query will return a single row with the total order amount per customer in 2024.
AFTER : SELECT SUM(amount) FROM orders WHERE cus

## 8. Save the Adapters

LoRA adapters are tiny(a few MB). You save only the adapter , not the whole base model .



In [21]:
ADAPTER_DIR = "./tinyllama-sql-lora-adapter"
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

import os
total = sum(
    os.path.getsize(os.path.join(ADAPTER_DIR, f))
    for f in os.listdir(ADAPTER_DIR)
    if os.path.isfile(os.path.join(ADAPTER_DIR, f))
)
print(f"Adapter size on disk: {total/1024**2:.2f} MB")

Adapter size on disk: 37.86 MB


In [23]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import torch

base = AutoModelForCausalLM.from_pretrained(
    "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    dtype=torch.float16,
    device_map="auto",
)
model = PeftModel.from_pretrained(base, "./tinyllama-sql-lora-adapter")
tokenizer = AutoTokenizer.from_pretrained("./tinyllama-sql-lora-adapter")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]